# 102 花卉数据集

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import datasets, transforms, models
import time
import copy
import os

In [2]:
data_dir = "./data/flower_data"
train_dir = data_dir + '/train'
valid_dir = data_dir + '/valid'

In [3]:
data_transforms = {
    "train": transforms.Compose([
        transforms.RandomRotation(45),
        transforms.CenterCrop(224),     # 从中心开始裁剪成 224x224 的图片
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.1, saturation=0.1, hue=0.1),  # 随机调整图片的亮度、对比度、饱和度、色调
        transforms.RandomGrayscale(0.025),  # 随机将图片转换为灰度图
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "valid": transforms.Compose([
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

In [4]:
batch_size = 32
image_datasets = {i: datasets.ImageFolder(os.path.join(data_dir, i), data_transforms[i]) for i in ['train', 'valid']}
data_loaders = {i: DataLoader(image_datasets[i], batch_size=batch_size, shuffle=True) for i in ['train', 'valid']}
dataset_sizes = {i: len(image_datasets[i]) for i in ['train', 'valid']}
class_names = image_datasets['train'].classes

In [5]:
import json

with open("./data/flower_data/cat_to_name.json", 'r') as f:
    cat_to_name = json.load(f)

In [6]:
def convert_tensor_to_image(tensor: torch.Tensor):
    image = tensor.to("cpu").clone().detach().numpy().squeeze().transpose(1, 2, 0)
    image = image * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    return image.clip(0, 1)

#### 迁移学习
迁移学习是指利用已有的模型，冻结其参数，只训练最后一层或几层全连接层，从而减少训练时间和资源开销。

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for param in model.parameters():
    param.requires_grad = False
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, len(class_names)),
    nn.LogSoftmax(dim=1)
)
image_size = 224
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [8]:
params_to_update = []
print("Params to learn:")
for name, param in model.named_parameters():
    if param.requires_grad:
        params_to_update.append(param)
        print("\t", name)

Params to learn:
	 fc.0.weight
	 fc.0.bias


In [9]:
optimizer = optim.Adam(params_to_update, lr=0.01)
lr_scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)
criterion = nn.NLLLoss()    # LogSoftmax + NLLLoss == CrossEntropyLoss

In [13]:
start_time = time.time()
best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
train_accuracy_history = []
valid_accuracy_history = []
train_loss_history = []
valid_loss_history = []
for epoch in range(10):
    train_time = time.time()
    train_loss = 0.0
    train_correct = 0
    model.train()
    for images, labels in data_loaders['train']:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        train_correct += (predicted == labels).sum().item()
    train_loss = train_loss / dataset_sizes['train']
    train_accuracy = 100.0 * train_correct / dataset_sizes['train']
    train_accuracy_history.append(train_accuracy)
    train_loss_history.append(train_loss)
    print(f"Epoch {epoch+1}/{20} | Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.2f}% | Time: {(time.time() - train_time):.2f}s")
    
    valid_time = time.time()
    valid_loss = 0.0
    valid_correct = 0
    model.eval()
    with torch.no_grad():
        for images, labels in data_loaders['valid']:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            valid_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            valid_correct += (predicted == labels).sum().item()
    valid_loss = valid_loss / dataset_sizes['valid']
    valid_accuracy = 100.0 * valid_correct / dataset_sizes['valid']
    valid_accuracy_history.append(valid_accuracy)
    valid_loss_history.append(valid_loss)
    lr_scheduler.step(valid_loss)
    print(f"Epoch {epoch+1}/{20} | Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_accuracy:.2f}% | Time: {(time.time() - valid_time):.2f}s")
    if valid_accuracy > best_acc:
        best_acc = valid_accuracy
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save({
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "accuracy": valid_accuracy,
        }, "./models/flower_classifier.pth")
print(f"Best Accuracy: {best_acc:.2f}%")
print(f"Training Time: {(time.time() - start_time):.2f}s")

Epoch 1/20 | Train Loss: 0.7259 | Train Acc: 84.14% | Time: 429.66s
Epoch 1/20 | Valid Loss: 0.7412 | Valid Acc: 87.04% | Time: 16.64s
Epoch 2/20 | Train Loss: 0.4758 | Train Acc: 88.23% | Time: 425.19s
Epoch 2/20 | Valid Loss: 0.6865 | Valid Acc: 87.53% | Time: 16.90s
Epoch 3/20 | Train Loss: 0.3888 | Train Acc: 90.26% | Time: 426.67s
Epoch 3/20 | Valid Loss: 0.6314 | Valid Acc: 86.80% | Time: 16.60s
Epoch 4/20 | Train Loss: 0.3304 | Train Acc: 91.61% | Time: 426.24s
Epoch 4/20 | Valid Loss: 0.5811 | Valid Acc: 89.00% | Time: 16.10s
Epoch 5/20 | Train Loss: 0.2490 | Train Acc: 92.98% | Time: 428.23s
Epoch 5/20 | Valid Loss: 0.5758 | Valid Acc: 89.61% | Time: 13.94s
Epoch 6/20 | Train Loss: 0.2140 | Train Acc: 93.85% | Time: 429.13s
Epoch 6/20 | Valid Loss: 0.5000 | Valid Acc: 90.22% | Time: 13.89s
Epoch 7/20 | Train Loss: 0.1944 | Train Acc: 94.38% | Time: 427.40s
Epoch 7/20 | Valid Loss: 0.4492 | Valid Acc: 90.59% | Time: 13.55s
Epoch 8/20 | Train Loss: 0.1518 | Train Acc: 95.63% | T

In [12]:
# 继续训练所有层
model.load_state_dict(best_model_wts)
for param in model.parameters():
    param.requires_grad = True
optimizer = optim.Adam(model.parameters(), lr=1e-4)
lr_scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)